# AUTOTROLEJ Live Data EDA (Minimal)

This notebook does one live snapshot to quickly understand realtime API data.

Scope:
1. Configure runtime and credentials from environment variables
2. Login and fetch one `/api/open/v1/voznired/autobusi` snapshot
3. Profile shape, nulls, duplicates, and coordinate ranges
4. Render a quick geographic view
5. Print short findings

## 1) Environment Setup and Imports

In [1]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Optional, List, Tuple
from urllib.parse import urljoin
from urllib.request import Request, urlopen

import pandas as pd

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")

Python: 3.14.3
pandas: 3.0.0


## 2) Configuration and Runtime Constants

In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "eda" else Path.cwd().resolve()
ENV_FILE = PROJECT_ROOT / ".env"

if ENV_FILE.exists():
    for raw_line in ENV_FILE.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

API_BASE_URL = os.getenv("ARRIVAL_API_BASE_URL", "").strip()
LOGIN_PATH = "/api/open/v1/token/login"
AUTOBUSI_PATH = "/api/open/v1/voznired/autobusi"
TIMEOUT_SECONDS = 30

ENV_USERNAME = "ARRIVAL_API_USERNAME"
ENV_PASSWORD = "ARRIVAL_API_PASSWORD"

LIVE_RAW_DIR = PROJECT_ROOT / "eda" / "data_raw" / "live"
LIVE_RAW_DIR.mkdir(parents=True, exist_ok=True)
SAVE_LIVE_RAW_SNAPSHOT = True

snapshot_time_utc = datetime.now(timezone.utc)

print(f"Loaded .env file: {ENV_FILE.exists()}")
print(f"API_BASE_URL set: {bool(API_BASE_URL)}")
print(f"Username env set: {bool(os.getenv(ENV_USERNAME))}")
print(f"Password env set: {bool(os.getenv(ENV_PASSWORD))}")
print(f"LIVE_RAW_DIR: {LIVE_RAW_DIR}")
print(f"SAVE_LIVE_RAW_SNAPSHOT: {SAVE_LIVE_RAW_SNAPSHOT}")
print(f"Snapshot UTC: {snapshot_time_utc.isoformat()}")

Loaded .env file: True
API_BASE_URL set: True
Username env set: True
Password env set: True
LIVE_RAW_DIR: /Users/ediprodan/Documents/Projects/arRIval/eda/data_raw/live
SAVE_LIVE_RAW_SNAPSHOT: True
Snapshot UTC: 2026-02-14T15:50:03.966762+00:00


## 3) API Helpers (Login + GET)

In [3]:
def api_get_json(url: str, headers: Optional[Dict[str, str]] = None, timeout: int = TIMEOUT_SECONDS):
    request = Request(url, headers=headers or {}, method="GET")
    with urlopen(request, timeout=timeout) as response:
        raw_text = response.read().decode("utf-8")
        return json.loads(raw_text)


def login_get_token(base_url: str, username: str, password: str) -> str:
    login_url = urljoin(base_url, LOGIN_PATH)
    payload = api_get_json(
        login_url,
        headers={
            "Username": username,
            "Password": password,
            "Accept": "application/json",
            "User-Agent": "arRIval-live-eda",
        },
    )
    if isinstance(payload, str) and payload.strip():
        return payload.strip()
    raise ValueError(f"Unexpected login response type: {type(payload)}")


def unwrap_wrapper(payload: dict) -> Tuple[Optional[str], Optional[bool], List[dict]]:
    msg = payload.get("msg") if isinstance(payload, dict) else None
    err = payload.get("err") if isinstance(payload, dict) else None
    res = payload.get("res") if isinstance(payload, dict) else None
    if res is None:
        return msg, err, []
    if not isinstance(res, list):
        raise ValueError(f"Expected wrapper 'res' list, got {type(res)}")
    return msg, err, res

## 4) Fetch One Live Snapshot (`/autobusi`)

In [4]:
if not API_BASE_URL:
    raise ValueError("Set ARRIVAL_API_BASE_URL before running.")

username = os.getenv(ENV_USERNAME, "").strip()
password = os.getenv(ENV_PASSWORD, "").strip()
if not username or not password:
    raise ValueError("Set ARRIVAL_API_USERNAME and ARRIVAL_API_PASSWORD before running.")

token = login_get_token(API_BASE_URL, username, password)
masked_token = f"{token[:6]}...{token[-4:]}" if len(token) >= 12 else "(short token)"
print(f"Token acquired: {masked_token}")

live_url = urljoin(API_BASE_URL, AUTOBUSI_PATH)
raw_payload = api_get_json(
    live_url,
    headers={
        "token": token,
        "Accept": "application/json",
        "User-Agent": "arRIval-live-eda",
    },
)

if SAVE_LIVE_RAW_SNAPSHOT:
    snapshot_file = LIVE_RAW_DIR / f"autobusi_{snapshot_time_utc.strftime('%Y%m%dT%H%M%SZ')}.json"
    with snapshot_file.open("w", encoding="utf-8") as f:
        json.dump(raw_payload, f, ensure_ascii=False)
    print(f"Saved raw snapshot: {snapshot_file.name}")

msg, err, records = unwrap_wrapper(raw_payload)
autobusi_df = pd.json_normalize(records)

print(f"Wrapper err: {err}")
print(f"Wrapper msg: {msg}")
print(f"Live rows: {len(autobusi_df)}")

display(autobusi_df.head(10))

Token acquired: n+bi5r...xw==
Saved raw snapshot: autobusi_20260214T155003Z.json
Wrapper err: False
Wrapper msg: ok
Live rows: 43


,gbr,lon,lat,voznjaId,voznjaBusId
0,838,14.275823,45.293137,1386940.0,2124287
1,784,14.302432,45.332847,1386924.0,2124314
2,830,14.447945,45.324783,1385745.0,2127881
3,787,14.440935,45.342292,1385979.0,2127646
4,826,0.000000,0.000000,NaN,2064950
5,778,14.386997,45.369395,1386813.0,2124436
6,799,14.465000,45.322082,1385678.0,2127946
7,840,14.448649,45.329262,NaN,2126569
8,664,14.479384,45.323518,1385927.0,2127679
9,790,14.397205,45.342735,1385870.0,2127733


## 5) Minimal Profiling

In [5]:
if autobusi_df.empty:
    profile_df = pd.DataFrame()
    print("No live records in this snapshot.")
else:
    profile_rows = []
    total_rows = len(autobusi_df)

    for col in autobusi_df.columns:
        null_count = int(autobusi_df[col].isna().sum())
        profile_rows.append(
            {
                "column": col,
                "dtype": str(autobusi_df[col].dtype),
                "null_count": null_count,
                "null_pct": round((null_count / total_rows) * 100, 2),
                "n_unique": int(autobusi_df[col].nunique(dropna=True)),
            }
        )

    profile_df = pd.DataFrame(profile_rows).sort_values("null_pct", ascending=False)

    duplicate_rows = int(autobusi_df.duplicated().sum())
    basic_stats = {
        "rows": total_rows,
        "columns": int(autobusi_df.shape[1]),
        "duplicate_rows": duplicate_rows,
        "duplicate_pct": round((duplicate_rows / total_rows) * 100, 2),
        "unique_gbr": int(autobusi_df["gbr"].nunique(dropna=True)) if "gbr" in autobusi_df.columns else None,
        "lat_min": float(autobusi_df["lat"].min()) if "lat" in autobusi_df.columns and autobusi_df["lat"].notna().any() else None,
        "lat_max": float(autobusi_df["lat"].max()) if "lat" in autobusi_df.columns and autobusi_df["lat"].notna().any() else None,
        "lon_min": float(autobusi_df["lon"].min()) if "lon" in autobusi_df.columns and autobusi_df["lon"].notna().any() else None,
        "lon_max": float(autobusi_df["lon"].max()) if "lon" in autobusi_df.columns and autobusi_df["lon"].notna().any() else None,
    }

    print("Basic stats")
    display(pd.DataFrame([basic_stats]))

    print("\nColumn profile")
    display(profile_df)

Basic stats


,rows,columns,duplicate_rows,duplicate_pct,unique_gbr,lat_min,lat_max,lon_min,lon_max
0,43,5,0,0.0,43,0.0,45.388747,0.0,14.598708



Column profile


,column,dtype,null_count,null_pct,n_unique
3,voznjaId,float64,10,23.26,33
0,gbr,int64,0,0.00,43
1,lon,float64,0,0.00,42
2,lat,float64,0,0.00,42
4,voznjaBusId,int64,0,0.00,43


## 6) Quick Geographic View

In [6]:
if autobusi_df.empty or not {"lat", "lon"}.issubset(autobusi_df.columns):
    print("Map skipped: missing rows or lat/lon columns.")
else:
    map_df = autobusi_df.dropna(subset=["lat", "lon"]).copy()
    map_df = map_df[(map_df["lat"] != 0) | (map_df["lon"] != 0)]

    if map_df.empty:
        print("Map skipped: all coordinates are missing/zero.")
    else:
        center_lat = float(map_df["lat"].median())
        center_lon = float(map_df["lon"].median())

        # Minimal enrichment: live voznjaBusId -> static PolazakId (dnevni only)
        map_df["voznjaBusId_int"] = pd.to_numeric(map_df.get("voznjaBusId"), errors="coerce")
        static_dir = PROJECT_ROOT / "eda" / "data_raw"
        dnevni_path = static_dir / "voznired_dnevni.json"

        if dnevni_path.exists():
            dnevni_raw = json.loads(dnevni_path.read_text(encoding="utf-8"))
            dnevni_df = pd.json_normalize(dnevni_raw)
            required_cols = {"PolazakId", "BrojLinije", "NazivVarijanteLinije", "Smjer"}

            if required_cols.issubset(dnevni_df.columns):
                dnevni_df = dnevni_df.copy()
                dnevni_df["PolazakId_int"] = pd.to_numeric(dnevni_df["PolazakId"], errors="coerce")
                lookup = (
                    dnevni_df.dropna(subset=["PolazakId_int"])
                    .sort_values("RedniBrojStanice")
                    .drop_duplicates("PolazakId_int")
                    [["PolazakId_int", "BrojLinije", "NazivVarijanteLinije", "Smjer"]]
                )

                map_df = map_df.merge(
                    lookup,
                    left_on="voznjaBusId_int",
                    right_on="PolazakId_int",
                    how="left",
                )
                matched_rows = int(map_df["BrojLinije"].notna().sum())
                match_pct = (matched_rows / len(map_df) * 100) if len(map_df) else 0
                print(f"Route enrichment source: voznired_dnevni (matched {matched_rows}/{len(map_df)} rows, {match_pct:.1f}%)")
            else:
                map_df["BrojLinije"] = None
                map_df["NazivVarijanteLinije"] = None
                map_df["Smjer"] = None
                print("Route enrichment unavailable: dnevni file missing required columns.")
        else:
            map_df["BrojLinije"] = None
            map_df["NazivVarijanteLinije"] = None
            map_df["Smjer"] = None
            print("Route enrichment unavailable: voznired_dnevni.json not found.")

        map_df["hover_title"] = map_df.apply(
            lambda r: f"Linija {r['BrojLinije']} | Bus {int(r['gbr'])}" if pd.notna(r.get("BrojLinije")) else f"Bus {int(r['gbr'])}",
            axis=1,
        )

        try:
            import plotly.express as px

            fig = px.scatter_map(
                map_df,
                lat="lat",
                lon="lon",
                hover_name="hover_title",
                hover_data={
                    "NazivVarijanteLinije": True,
                    "Smjer": True,
                    "voznjaId": True,
                    "voznjaBusId": True,
                    "lat": False,
                    "lon": False,
                },
                zoom=12,
                center={"lat": center_lat, "lon": center_lon},
                title="Live bus positions (enriched with line + route)",
                height=600,
            )
            fig.update_layout(map_style="carto-positron", margin={"l": 0, "r": 0, "t": 40, "b": 0})
            fig.show()
        except Exception as plotly_error:
            print(f"Plotly map skipped: {plotly_error}")
            try:
                import matplotlib.pyplot as plt

                plt.figure(figsize=(7, 6))
                plt.scatter(map_df["lon"], map_df["lat"], s=20)
                plt.title("Live bus positions (enriched with line + route)")
                plt.xlabel("Longitude")
                plt.ylabel("Latitude")
                plt.grid(alpha=0.2)
                plt.tight_layout()
                plt.show()
            except Exception as mpl_error:
                print(f"Matplotlib fallback skipped: {mpl_error}")

Route enrichment source: voznired_dnevni (matched 33/41 rows, 80.5%)


## 7) Minimal Validation and Findings

In [7]:
if err is False:
    assert isinstance(records, list), "Expected wrapper res to be a list when err=False."

if not autobusi_df.empty and {"lat", "lon"}.issubset(autobusi_df.columns):
    coord_df = autobusi_df.dropna(subset=["lat", "lon"])
    if not coord_df.empty:
        valid_lat = coord_df["lat"].between(-90, 90).all()
        valid_lon = coord_df["lon"].between(-180, 180).all()
        assert valid_lat and valid_lon, "Coordinates out of global bounds."

print("Live EDA snapshot findings\n")
print(f"- Snapshot time (UTC): {snapshot_time_utc.isoformat()}")
print(f"- Wrapper err flag: {err}")
print(f"- Live rows: {len(autobusi_df)}")
if "gbr" in autobusi_df.columns:
    print(f"- Unique buses (gbr): {autobusi_df['gbr'].nunique(dropna=True)}")
if "lat" in autobusi_df.columns and "lon" in autobusi_df.columns and len(autobusi_df) > 0:
    non_null_coords = autobusi_df[["lat", "lon"]].notna().all(axis=1).mean() * 100
    print(f"- Rows with both coordinates: {non_null_coords:.2f}%")
if not autobusi_df.empty:
    duplicate_rows = int(autobusi_df.duplicated().sum())
    print(f"- Duplicate rows in snapshot: {duplicate_rows}")

print("\nDone. This is intentionally minimal: one-call live data understanding.")

Live EDA snapshot findings

- Snapshot time (UTC): 2026-02-14T15:50:03.966762+00:00
- Wrapper err flag: False
- Live rows: 43
- Unique buses (gbr): 43
- Rows with both coordinates: 100.00%
- Duplicate rows in snapshot: 0

Done. This is intentionally minimal: one-call live data understanding.


## 8) Minimal Data Interpretation

In [8]:
if autobusi_df.empty:
    print("Interpretation skipped: no live rows.")
else:
    total = len(autobusi_df)

    zero_coord_mask = (
        autobusi_df["lat"].fillna(0).eq(0) & autobusi_df["lon"].fillna(0).eq(0)
        if {"lat", "lon"}.issubset(autobusi_df.columns)
        else pd.Series([False] * total)
    )
    zero_coord_rows = int(zero_coord_mask.sum())

    missing_voznja = int(autobusi_df["voznjaId"].isna().sum()) if "voznjaId" in autobusi_df.columns else None

    active_rows = int((~autobusi_df["voznjaId"].isna()).sum()) if "voznjaId" in autobusi_df.columns else None
    inactive_rows = int(autobusi_df["voznjaId"].isna().sum()) if "voznjaId" in autobusi_df.columns else None

    print("Quick interpretation")
    print(f"- Total live rows: {total}")
    print(f"- Zero-coordinate rows (0,0): {zero_coord_rows} ({(zero_coord_rows / total) * 100:.2f}%)")

    if missing_voznja is not None:
        print(f"- Rows without voznjaId: {missing_voznja} ({(missing_voznja / total) * 100:.2f}%)")
        print(f"- Rows with active trip ID: {active_rows}")
        print(f"- Rows likely not in active trip: {inactive_rows}")

    if "map_df" in globals() and isinstance(map_df, pd.DataFrame) and "BrojLinije" in map_df.columns:
        mapped_rows = int(map_df["BrojLinije"].notna().sum())
        unmapped_rows = int(len(map_df) - mapped_rows)
        mapped_pct = (mapped_rows / len(map_df) * 100) if len(map_df) else 0.0
        print(f"- Route mapping in map view: {mapped_rows}/{len(map_df)} ({mapped_pct:.2f}%), unmapped={unmapped_rows}")

        if mapped_rows > 0:
            print("- Top 5 active lines in snapshot:")
            display(map_df.loc[map_df["BrojLinije"].notna(), "BrojLinije"].value_counts().head(5).to_frame("buses"))

    print("\nMinimal takeaways")
    print("- Treat (0,0) coordinates as invalid/no-position rows.")
    print("- `voznjaId` nulls likely represent parked/transition/off-schedule buses.")
    print("- Snapshot shape is stable enough for next minimal step: short polling window.")

Quick interpretation
- Total live rows: 43
- Zero-coordinate rows (0,0): 2 (4.65%)
- Rows without voznjaId: 10 (23.26%)
- Rows with active trip ID: 33
- Rows likely not in active trip: 10
- Route mapping in map view: 33/41 (80.49%), unmapped=8
- Top 5 active lines in snapshot:


,buses
BrojLinije,
2,4
32,3
7,2
6,2
26,2



Minimal takeaways
- Treat (0,0) coordinates as invalid/no-position rows.
- `voznjaId` nulls likely represent parked/transition/off-schedule buses.
- Snapshot shape is stable enough for next minimal step: short polling window.
